In [ ]:
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import (
    Font, PatternFill, Alignment, Border, Side, GradientFill
)
from openpyxl.utils import get_column_letter
from openpyxl.drawing.image import Image as XLImage
import warnings
warnings.filterwarnings('ignore')

BASE = r'C:\Users\WELCOME\Desktop\DataAnalysis_Projects\ab-testing-marketing-analysis'

print("All libraries loaded successfully")

In [ ]:
df       = pd.read_csv(BASE + r'\data\marketing_AB.csv')
results  = pd.read_csv(BASE + r'\data\hypothesis_test_results.csv')
eda      = pd.read_csv(BASE + r'\data\eda_summary.csv')

ad  = df[df['test group'] == 'ad']
psa = df[df['test group'] == 'psa']

rate_ad  = ad['converted'].mean()
rate_psa = psa['converted'].mean()

print("Data loaded successfully")
print(f"Results rows: {len(results)}")

In [ ]:
wb = Workbook()

# Colour palette
DARK_BLUE   = "1A3A5C"
MID_BLUE    = "2E6DA4"
LIGHT_BLUE  = "D6E4F0"
GREEN       = "1D9E75"
LIGHT_GREEN = "D5F0E8"
RED         = "C0392B"
LIGHT_RED   = "FADBD8"
WHITE       = "FFFFFF"
LIGHT_GREY  = "F5F5F5"
DARK_GREY   = "333333"

def style_header(cell, bg=DARK_BLUE, fg=WHITE, size=11, bold=True):
    cell.font      = Font(bold=bold, color=fg, size=size, name='Arial')
    cell.fill      = PatternFill("solid", fgColor=bg)
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

def style_cell(cell, bold=False, size=10, color=DARK_GREY, bg=None, align='left'):
    cell.font      = Font(bold=bold, color=color, size=size, name='Arial')
    cell.alignment = Alignment(horizontal=align, vertical='center', wrap_text=True)
    if bg:
        cell.fill  = PatternFill("solid", fgColor=bg)

def add_border(cell):
    thin = Side(style='thin', color="CCCCCC")
    cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)

def set_col_width(ws, col, width):
    ws.column_dimensions[get_column_letter(col)].width = width

def set_row_height(ws, row, height):
    ws.row_dimensions[row].height = height

print("Helper functions defined")

In [ ]:
ws1 = wb.active
ws1.title = "Executive Summary"

# Title block
ws1.merge_cells('A1:F1')
ws1['A1'] = "A/B TEST REPORT — MARKETING CAMPAIGN ANALYSIS"
style_header(ws1['A1'], bg=DARK_BLUE, fg=WHITE, size=14)
set_row_height(ws1, 1, 36)

ws1.merge_cells('A2:F2')
ws1['A2'] = "Prepared by: Prabin Pokhrel  |  Dataset: Marketing A/B Testing  |  Sample: 588,101 users"
style_cell(ws1['A2'], size=9, color="666666", align='center')
ws1['A2'].fill = PatternFill("solid", fgColor="F0F0F0")
set_row_height(ws1, 2, 18)

# Section: Test Overview
ws1.merge_cells('A4:F4')
ws1['A4'] = "TEST OVERVIEW"
style_header(ws1['A4'], bg=MID_BLUE, size=10)
set_row_height(ws1, 4, 22)

overview = [
    ("Test Type",        "Two-sample proportion test (A/B)"),
    ("Control Group",    "PSA — Public Service Announcement (no ad)"),
    ("Test Group",       "Ad — Paid advertisement"),
    ("Total Users",      "588,101"),
    ("Test Duration",    "Full dataset (no time column available)"),
    ("Primary Metric",   "Conversion rate (user converted = 1)"),
    ("Alpha Level",      "0.05 (95% confidence)"),
    ("Statistical Tests","Chi-Square test + Two-proportion Z-test"),
]

for i, (label, value) in enumerate(overview, start=5):
    ws1[f'A{i}'] = label
    ws1[f'B{i}'] = value
    style_cell(ws1[f'A{i}'], bold=True, bg=LIGHT_BLUE if i%2==0 else WHITE)
    style_cell(ws1[f'B{i}'], bg=LIGHT_BLUE if i%2==0 else WHITE)
    ws1.merge_cells(f'B{i}:F{i}')
    add_border(ws1[f'A{i}'])
    add_border(ws1[f'B{i}'])
    set_row_height(ws1, i, 18)

# Section: KPI Cards
row = 14
ws1.merge_cells(f'A{row}:F{row}')
ws1[f'A{row}'] = "KEY METRICS"
style_header(ws1[f'A{row}'], bg=MID_BLUE, size=10)
set_row_height(ws1, row, 22)

kpis = [
    ("Ad Conversion Rate",  "2.5547%",  MID_BLUE,  WHITE),
    ("PSA Conversion Rate", "1.7854%",  DARK_GREY, WHITE),
    ("Absolute Uplift",     "+0.7692pp", GREEN,     WHITE),
    ("Relative Uplift",     "+43.09%",  GREEN,     WHITE),
    ("Z-Statistic",         "7.37",     DARK_BLUE, WHITE),
    ("P-Value",             "< 0.0001", GREEN,     WHITE),
]

col_positions = [1, 2, 3, 4, 5, 6]
for j, ((label, value, bg, fg), col) in enumerate(zip(kpis, col_positions)):
    label_row = row + 1
    value_row = row + 2
    ws1.cell(label_row, col).value = label
    ws1.cell(value_row, col).value = value
    style_header(ws1.cell(label_row, col), bg=bg, fg=fg, size=9)
    style_header(ws1.cell(value_row, col), bg=bg, fg=fg, size=13)
    set_row_height(ws1, label_row, 20)
    set_row_height(ws1, value_row, 28)

# Section: Statistical Results
row = 18
ws1.merge_cells(f'A{row}:F{row}')
ws1[f'A{row}'] = "STATISTICAL TEST RESULTS"
style_header(ws1[f'A{row}'], bg=MID_BLUE, size=10)
set_row_height(ws1, row, 22)

stat_results = [
    ("Chi-Square Statistic", "54.0058",          "Chi-Square P-Value",   "0.000000"),
    ("Z-Statistic",          "7.3701",            "Z-Test P-Value",       "0.000000"),
    ("CI Lower (95%)",       "0.5951%",           "CI Upper (95%)",       "0.9434%"),
    ("Cohen's h",            "0.0530 (Small)",    "Significant?",         "YES"),
]

headers = ["Metric", "Value", "Metric", "Value"]
for j, h in enumerate(headers, start=1):
    ws1.cell(row+1, j).value = h
    style_header(ws1.cell(row+1, j), bg=DARK_BLUE, size=9)
    set_row_height(ws1, row+1, 18)

for i, (m1, v1, m2, v2) in enumerate(stat_results, start=row+2):
    bg = LIGHT_GREY if i % 2 == 0 else WHITE
    for col, val in enumerate([m1, v1, m2, v2], start=1):
        ws1.cell(i, col).value = val
        style_cell(ws1.cell(i, col), bold=(col in [1,3]), bg=bg)
        add_border(ws1.cell(i, col))
    set_row_height(ws1, i, 18)

# Section: Recommendation
row = 27
ws1.merge_cells(f'A{row}:F{row}')
ws1[f'A{row}'] = "RECOMMENDATION"
style_header(ws1[f'A{row}'], bg=GREEN, size=10)
set_row_height(ws1, row, 22)

rec_text = (
    "DEPLOY THE AD CAMPAIGN. The A/B test provides strong statistical evidence "
    "(p < 0.0001, Z = 7.37) that the ad group converts at a significantly higher rate "
    "than the PSA control group (2.55% vs 1.79%). While Cohen's h = 0.053 indicates a "
    "small practical effect size — common with large samples — the 43% relative uplift "
    "and 95% CI entirely above zero confirm the ad's positive impact. At 50,000 monthly "
    "users, the ad is estimated to generate approximately 385 additional conversions per "
    "month versus the PSA alternative. Monday and Tuesday show the highest uplift and "
    "should be prioritised for ad delivery."
)

ws1.merge_cells(f'A{row+1}:F{row+3}')
ws1[f'A{row+1}'] = rec_text
ws1[f'A{row+1}'].font      = Font(size=10, name='Arial', color=DARK_GREY)
ws1[f'A{row+1}'].alignment = Alignment(wrap_text=True, vertical='top')
ws1[f'A{row+1}'].fill      = PatternFill("solid", fgColor=LIGHT_GREEN)
set_row_height(ws1, row+1, 60)

# Column widths
set_col_width(ws1, 1, 22)
for c in range(2, 7):
    set_col_width(ws1, c, 18)

print("Sheet 1: Executive Summary done")

In [ ]:
ws2 = wb.create_sheet("Statistical Results")

ws2.merge_cells('A1:B1')
ws2['A1'] = "FULL STATISTICAL RESULTS"
style_header(ws2['A1'], bg=DARK_BLUE, size=12)
set_row_height(ws2, 1, 30)

for col, header in enumerate(['Metric', 'Value'], start=1):
    ws2.cell(2, col).value = header
    style_header(ws2.cell(2, col), bg=MID_BLUE, size=10)
    set_row_height(ws2, 2, 20)

for i, row in results.iterrows():
    bg = LIGHT_GREY if i % 2 == 0 else WHITE
    ws2.cell(i+3, 1).value = row['Metric']
    ws2.cell(i+3, 2).value = row['Value']
    style_cell(ws2.cell(i+3, 1), bold=True, bg=bg)
    style_cell(ws2.cell(i+3, 2), bg=bg, align='center')
    add_border(ws2.cell(i+3, 1))
    add_border(ws2.cell(i+3, 2))
    set_row_height(ws2, i+3, 18)

set_col_width(ws2, 1, 30)
set_col_width(ws2, 2, 25)

print("Sheet 2: Statistical Results done")

In [ ]:
ws3 = wb.create_sheet("Segmentation")

ws3.merge_cells('A1:E1')
ws3['A1'] = "CONVERSION RATE SEGMENTATION BY DAY"
style_header(ws3['A1'], bg=DARK_BLUE, size=12)
set_row_height(ws3, 1, 30)

dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

seg = df.groupby(['most ads day','test group'])['converted'].agg(['sum','count','mean']).reset_index()
seg.columns = ['Day','Group','Conversions','Users','Rate']
seg['Rate %'] = (seg['Rate']*100).round(4)
seg['Day'] = pd.Categorical(seg['Day'], categories=dow_order, ordered=True)
seg = seg.sort_values(['Day','Group']).reset_index(drop=True)

headers = ['Day','Group','Users','Conversions','Conversion Rate %']
for col, h in enumerate(headers, start=1):
    ws3.cell(2, col).value = h
    style_header(ws3.cell(2, col), bg=MID_BLUE, size=10)
    set_row_height(ws3, 2, 20)

for i, row in seg.iterrows():
    r = i + 3
    bg = LIGHT_GREEN if row['Group'] == 'ad' else LIGHT_GREY
    for col, val in enumerate([row['Day'], row['Group'], row['Users'],
                                 row['Conversions'], row['Rate %']], start=1):
        ws3.cell(r, col).value = val
        style_cell(ws3.cell(r, col), bg=bg, align='center' if col > 1 else 'left')
        add_border(ws3.cell(r, col))
    set_row_height(ws3, r, 18)

for c in range(1, 6):
    set_col_width(ws3, c, 20)

print("Sheet 3: Segmentation done")

In [ ]:
output_path = BASE + r'\outputs\ab_test_report.xlsx'
wb.save(output_path)
print(f"Excel report saved: {output_path}")
print("\nSheets created:")
for sheet in wb.sheetnames:
    print(f"  {sheet}")